## NEW EXPERIMENTS ANALYSIS 

In [ ]:
import os
import pandas as pd
import itertools

# Folder containing the TCN CSV files
folder_path_lstm = "Window_size_and_Ratio_experiments_LSTM/"
folder_path_tcn = "Window_size_and_Ratio_experiments_TCN/"
folder_path_transformers = "Window_size_and_Ratio_experiments_Transformers/"
# General setup
window_size = 3  
ratio = 0.3 
iterations = range(1, 11)  # 1 to 10


# Store results per hyperparameter combination
tcn_results = {}
summary_table_tcn = []

# Loop over hyperparameter combinations
for filters, ks, dr, nb in itertools.product(
        tcn_grid["filters"], 
        tcn_grid["kernel_size"], 
        tcn_grid["dropout"], 
        tcn_grid["num_blocks"]):

    all_f1_scores = []

    for iteration in iterations:
        file_name = f"experiments_{window_size}_{ratio}_{iteration}_model_3_{filters}_{ks}_{nb}_{dr}.csv"
        file_path = os.path.join(folder_path_tcn, file_name)

        if not os.path.exists(file_path):
            continue

        df = pd.read_csv(file_path)

        # Filter rows where "Evaluation Type" is "Cumulative"
        df_cumulative = df[df["Evaluation Type"] == "Cumulative"].copy()

        # Compute F1-score safely
        df_cumulative["F1-score"] = (
            2 * (df_cumulative["Precision"] * df_cumulative["Recall"]) /
            (df_cumulative["Precision"] + df_cumulative["Recall"]).replace(0, pd.NA)
        )

        all_f1_scores.append(df_cumulative[["Day", "F1-score"]])

    if all_f1_scores:
        avg_f1_df = pd.concat(all_f1_scores).groupby("Day").mean().reset_index()
        tcn_results[(filters, ks, nb, dr)] = avg_f1_df

        # Extract F1 for Day 5 only
        f1_day5 = avg_f1_df.loc[avg_f1_df["Day"] == 5, "F1-score"]
        if not f1_day5.empty:
            summary_table_tcn.append((filters, ks, nb, dr, f1_day5.values[0]))

# Print results for each TCN hyperparameter combination
for (filters, ks, nb, dr), df in tcn_results.items():
    print(f"\n=== Results for TCN (filters={filters}, kernel_size={ks}, num_blocks={nb}, dropout={dr}) ===")
    print(df)

# Create summary DataFrame
if summary_table_tcn:
    summary_df_tcn = pd.DataFrame(
        summary_table_tcn, 
        columns=["filters", "kernel_size", "num_blocks", "dropout", "F1_day5"]
    )
    print("\n=== Summary of TCN Hyperparameter Combinations (Day 5 only) ===")
    print(summary_df_tcn.sort_values(by="F1_day5", ascending=False).reset_index(drop=True))

    # Find the best combo (Day 5 F1-score)
    best_row = summary_df_tcn.loc[summary_df_tcn["F1_day5"].idxmax()]
    print("\n=== Best TCN Combination (Day 5) ===")
    print(f"filters={best_row['filters']}, kernel_size={best_row['kernel_size']}, "
          f"num_blocks={best_row['num_blocks']}, dropout={best_row['dropout']}, "
          f"Day 5 F1={best_row['F1_day5']:.4f}")


In [ ]:
import os
import pandas as pd
import itertools

# Folder containing the CSV files
folder_path = "Comparison_Experiments_LSTM_TCN_Transformers/Finetuning_Transformers/"

# Define parameter ranges
window_size = 3  
ratio = 0.3 
iterations = range(1, 11)  # 1 to 10
head_sizes = [32, 64]
num_heads = [2, 4]
ff_dims = [64, 128]
dropouts = [0.1, 0.2]

# Store results per hyperparameter combination
transformer_results = {}
summary_table = []

# Loop over hyperparameter combinations
for hs, nh, ff, dr in itertools.product(head_sizes, num_heads, ff_dims, dropouts):
    all_f1_scores = []  # store F1-scores across iterations for this combo

    for iteration in iterations:
        file_name = f"experiments_{window_size}_{ratio}_{iteration}_model_1_{hs}_{nh}_{ff}_{dr}.csv"
        file_path = os.path.join(folder_path, file_name)

        if not os.path.exists(file_path):
            continue

        df = pd.read_csv(file_path)

        # Filter rows where "Evaluation Type" is "Cumulative"
        df_cumulative = df[df["Evaluation Type"] == "Cumulative"].copy()

        # Compute F1-score safely
        df_cumulative["F1-score"] = (
            2 * (df_cumulative["Precision"] * df_cumulative["Recall"]) /
            (df_cumulative["Precision"] + df_cumulative["Recall"]).replace(0, pd.NA)
        )

        all_f1_scores.append(df_cumulative[["Day", "F1-score"]])

    # Average across iterations if results exist
    if all_f1_scores:
        avg_f1_df = pd.concat(all_f1_scores).groupby("Day").mean().reset_index()
        transformer_results[(hs, nh, ff, dr)] = avg_f1_df

        # Extract F1 for Day 5 only (if exists)
        f1_day5 = avg_f1_df.loc[avg_f1_df["Day"] == 5, "F1-score"]
        if not f1_day5.empty:
            summary_table.append((hs, nh, ff, dr, f1_day5.values[0]))

# Print results for each Transformer hyperparameter combination
for (hs, nh, ff, dr), df in transformer_results.items():
    print(f"\n=== Results for Transformer (hs={hs}, num_heads={nh}, ff_dim={ff}, dropout={dr}) ===")
    print(df)

# Create summary DataFrame
if summary_table:
    summary_df = pd.DataFrame(summary_table, columns=["head_size", "num_heads", "ff_dim", "dropout", "F1_day5"])
    print("\n=== Summary of Transformer Hyperparameter Combinations (Day 5 only) ===")
    print(summary_df.sort_values(by="F1_day5", ascending=False).reset_index(drop=True))

    # Find the best combo (based only on Day 5 F1-score)
    best_row = summary_df.loc[summary_df["F1_day5"].idxmax()]
    print("\n=== Best Transformer Combination (Day 5) ===")
    print(f"head_size={best_row['head_size']}, num_heads={best_row['num_heads']}, "
          f"ff_dim={best_row['ff_dim']}, dropout={best_row['dropout']}, "
          f"Day 5 F1={best_row['F1_day5']:.4f}")


In [ ]:
import os
import pandas as pd

# Settings
folder_path = "Comparison_Experiments_LSTM_TCN_Transformers"
window_sizes = 3   # or more if needed
ratio = 0.3
iterations = range(1, 11)  # 1 to 10
model_choices = [0, 1, 3]  

# Store results: dict[model_choice] = DataFrame of metrics
fpr_results = {}
fnr_results = {}

for model_choice in model_choices:
    # Temp storage for this model
    model_fpr = {}
    model_fnr = {}

    
    fpr_values = []
    fnr_values = []

    for iteration in iterations:
        file_name = f"experiments_{window_size}_{ratio}_{iteration}_model_{model_choice}.csv"
        file_path = os.path.join(folder_path, file_name)

        if not os.path.exists(file_path):
            continue

        df = pd.read_csv(file_path)

        # Filter for Day 5 and "Cumulative" evaluation type
        df_filtered = df[(df["Day"] == 5) & (df["Evaluation Type"] == "Cumulative")].copy()
        if df_filtered.empty:
            continue

        try:
            # Parse confusion matrix
            df_filtered["Confusion Matrix"] = df_filtered["Confusion Matrix"].apply(eval)
            df_filtered["TN"] = df_filtered["Confusion Matrix"].apply(lambda x: x[0][0])
            df_filtered["FP"] = df_filtered["Confusion Matrix"].apply(lambda x: x[0][1])
            df_filtered["FN"] = df_filtered["Confusion Matrix"].apply(lambda x: x[1][0])
            df_filtered["TP"] = df_filtered["Confusion Matrix"].apply(lambda x: x[1][1])

            # Compute FPR and FNR
            df_filtered["FPR"] = df_filtered["FP"] / (df_filtered["FP"] + df_filtered["TN"])
            df_filtered["FNR"] = df_filtered["FN"] / (df_filtered["FN"] + df_filtered["TP"])

            fpr_values.append(df_filtered["FPR"].values[0])
            fnr_values.append(df_filtered["FNR"].values[0])

        except Exception as e:
            print(f"Error processing {file_name}: {e}")

    # Average across iterations
    avg_fpr = sum(fpr_values) / len(fpr_values) * 100 if fpr_values else None
    avg_fnr = sum(fnr_values) / len(fnr_values) * 100 if fnr_values else None

    fpr_results[model_choice] = avg_fpr
    fnr_results[model_choice] = avg_fnr

# Print results
model_names = {0: "LSTM", 1: "Transformer", 3: "TCN"}

for mc in model_choices:
    print(f"\n=== {model_names[mc]} (model_choice={mc}) ===")
    if fpr_results[mc] is not None:
        print(f"Average False Positive Rate (FPR): {fpr_results[mc]:.2f}%")
    else:
        print("Average False Positive Rate (FPR): N/A")

    if fnr_results[mc] is not None:
        print(f"Average False Negative Rate (FNR): {fnr_results[mc]:.2f}%")
    else:
        print("Average False Negative Rate (FNR): N/A")


In [ ]:
import os
import pandas as pd
import ast  # for safely parsing the dict stored as string

# Folder containing the LSTM experiment files
folder_path = "Window_size_and_Ratio_experiments_LSTM"

# Define parameter ranges
window_sizes = range(1, 11)  # 1 to 10
ratios = [round(i * 0.1, 1) for i in range(6)]  # 0.0 to 0.5
iterations = range(0, 10)  # 0 to 9

# Dictionary to store final DataFrames
final_results = {}
# List to store Day 5 cumulative values for ranking
ranking_results = []

def compute_fpr_fnr(cm):
    """Compute False Positive Rate and False Negative Rate in % from confusion matrix"""
    tn, fp = cm[0]
    fn, tp = cm[1]
    fpr = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0
    fnr = (fn / (fn + tp)) * 100 if (fn + tp) > 0 else 0
    return fpr, fnr

# Loop over window sizes
for window_size in window_sizes:
    for ratio in ratios:
        all_metrics = []  # Store metrics for averaging

        # Loop over iterations
        for iteration in iterations:
            file_name = f"experiments_{window_size}_{ratio}_iter_{iteration}.csv"
            file_path = os.path.join(folder_path, file_name)

            if not os.path.exists(file_path):
                print(f"Warning: {file_path} not found. Skipping...")
                continue
            
            # Read CSV
            df = pd.read_csv(file_path)

            # Filter rows where "Evaluation Type" is "Cumulative"
            df_cumulative = df[df["Evaluation Type"] == "Cumulative"].copy()

            # Parse classification report and confusion matrix
            df_cumulative["Report"] = df_cumulative["Classification Report"].apply(ast.literal_eval)
            df_cumulative["Confusion"] = df_cumulative["Confusion Matrix"].apply(ast.literal_eval)

            # Extract metrics for each row
            metrics_list = []
            for _, row in df_cumulative.iterrows():
                report = row["Report"]
                cm = row["Confusion"]
                fpr, fnr = compute_fpr_fnr(cm)

                metrics_list.append({
                    "Day": row["Day"],
                    # Micro-style F1
                    "F1-score": 2 * (row["Precision"] * row["Recall"]) / (row["Precision"] + row["Recall"]),
                    # Accuracy from report
                    "Accuracy": report["accuracy"],
                    # Macro avg
                    "Macro Precision": report["macro avg"]["precision"],
                    "Macro Recall": report["macro avg"]["recall"],
                    "Macro F1": report["macro avg"]["f1-score"],
                    # Weighted avg
                    "Weighted Precision": report["weighted avg"]["precision"],
                    "Weighted Recall": report["weighted avg"]["recall"],
                    "Weighted F1": report["weighted avg"]["f1-score"],
                    # Confusion-based
                    "FPR (%)": fpr,
                    "FNR (%)": fnr
                })
            
            all_metrics.append(pd.DataFrame(metrics_list))

        # If we gathered results, compute averages across iterations
        if all_metrics:
            avg_df = pd.concat(all_metrics).groupby("Day").mean().reset_index()
            final_results[(window_size, ratio)] = avg_df

            # Extract Day 5 row for ranking
            day5_row = avg_df[avg_df["Day"] == 5]
            if not day5_row.empty:
                ranking_results.append({
                    "Window Size": window_size,
                    "Ratio": ratio,
                    "Day 5 F1-score": day5_row["F1-score"].values[0],
                    "Day 5 Accuracy": day5_row["Accuracy"].values[0],
                    "Day 5 Macro F1": day5_row["Macro F1"].values[0],
                    "Day 5 Weighted F1": day5_row["Weighted F1"].values[0],
                    "Day 5 FPR (%)": day5_row["FPR (%)"].values[0],
                    "Day 5 FNR (%)": day5_row["FNR (%)"].values[0]
                })

# Display results (all days)
for key, df in final_results.items():
    window_size, ratio = key
    print(f"\n=== LSTM - Window Size: {window_size}, Ratio: {ratio} ===")
    print(df.to_string(index=False))

# Build leaderboard DataFrame
df_ranking = pd.DataFrame(ranking_results)
df_ranking_sorted = df_ranking.sort_values(by="Day 5 F1-score", ascending=False).reset_index(drop=True)

# Print leaderboard
print("\n=== LSTM Leaderboard ranked by Day 5 Cumulative F1-score ===\n")
print(df_ranking_sorted.to_string(index=False))


In [ ]:
# Filter only Day 5 rows
df_day5 = df_ranking.copy()

# Aggregate Macro F1 for Ratio = 0.0
agg_no_ratio = (
    df_day5[df_day5["Ratio"] == 0.0]
    .groupby("Window Size")["Day 5 Macro F1"]
    .mean()
)

# Aggregate Macro F1 for all Ratios ≠ 0.0
agg_with_ratio = (
    df_day5[df_day5["Ratio"] != 0.0]
    .groupby("Window Size")["Day 5 Macro F1"]
    .mean()
)

# Unique ratios for the second subplot (exclude 0.0)
unique_ratios = sorted(df_day5[df_day5["Ratio"] != 0.0]["Ratio"].unique())


In [ ]:
import matplotlib.pyplot as plt
colors = ["green", "orange", "purple", "brown", "pink"]  # example colors
marker_styles = ["o", "s", "D", "^", "v"]                # example markers

# Create a single figure with two subplots and different y-axis ticks
fig, axes = plt.subplots(1, 2, figsize=(6, 2.5), dpi=300, sharey=False)

# First subplot: Ratio = 0.0 vs Ratio ≠ 0.0 with unique y-axis ticks
ax1 = axes[0]
line1, = ax1.plot(
    agg_no_ratio.index, agg_no_ratio.values, linestyle="--", marker="o", label="Ratio = 0.0",
    color="red", linewidth=2, markersize=5
)
line2, = ax1.plot(
    agg_with_ratio.index, agg_with_ratio.values, linestyle="-", marker="s", label="Avg. With Ratio (≠ 0.0)",
    color="blue", linewidth=2, markersize=5
)
ax1.set_xlabel("Window Size", fontsize=12)
ax1.set_ylabel("Macro F1-Score", fontsize=12)
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.set_xticks(sorted(df_day5["Window Size"].unique()))
ax1.set_yticks([0.7, 0.8, 0.9, 1.00])  # Custom y-axis ticks for first plot
ax1.tick_params(axis='both', labelsize=10)

# Second subplot: Comparing Ratios from 0.1 to 0.5 with unique y-axis ticks
ax2 = axes[1]
# Lock the y-axis from 0.94 to 1.00
ax2.set_ylim(0.94, 0.98)

# (optional) nice ticks to match the limits
ax2.set_yticks([0.94, 0.96, 0.98])
lines = []
for (ratio, color, marker) in zip(unique_ratios, colors, marker_styles):
    df_ratio = df_day5[df_day5["Ratio"] == ratio]
    aggregated_f1 = df_ratio.groupby("Window Size")["Day 5 Macro F1"].mean()
    line, = ax2.plot(
        aggregated_f1.index, aggregated_f1.values, linestyle="-", marker=marker,
        label=f"Ratio {ratio:.1f}", color=color, linewidth=2, markersize=5
    )
    lines.append(line)

ax2.set_xlabel("Window Size", fontsize=12)
ax2.grid(True, linestyle="--", alpha=0.6)
ax2.set_xticks(sorted(df_day5["Window Size"].unique()))
#ax2.set_yticks([0.90, 0.92, 0.94, 0.96])  # Custom y-axis ticks for second plot
ax2.tick_params(axis='both', labelsize=10)

# Common legend below both plots
fig.legend(handles=[line1, line2] + lines, loc="lower center", bbox_to_anchor=(0.5, -0.3), ncol=3, fontsize=10)

# Adjust layout
plt.tight_layout()

# Save the combined figure
pdf_path_combined = "Day5_Combined_Figure.pdf"
fig.savefig(pdf_path_combined, format="pdf", bbox_inches="tight")


In [ ]:
import os
import pandas as pd
import ast  # to safely parse the Classification Report and Confusion Matrix dicts

# Folder containing the Transformer experiment files
folder_path = "Window_size_and_Ratio_experiments_Transformers"

# Define parameter ranges
window_sizes = range(1, 11)  # 1 to 10
ratios = [round(i * 0.1, 1) for i in range(6)]  # 0.0 to 0.5
iterations = range(1, 4)  # 1 to 3

# Dictionary to store per-day averages
final_results = {}
# List to store Day 5 cumulative averages for ranking
ranking_results = []

def compute_fpr_fnr(cm):
    """Compute False Positive Rate and False Negative Rate in % from confusion matrix"""
    tn, fp = cm[0]
    fn, tp = cm[1]
    fpr = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0
    fnr = (fn / (fn + tp)) * 100 if (fn + tp) > 0 else 0
    return fpr, fnr

# Loop over window sizes
for window_size in window_sizes:
    for ratio in ratios:
        all_metrics = []  # Store metrics for averaging

        # Loop over iterations
        for iteration in iterations:
            file_name = f"experiments_{window_size}_{ratio}_{iteration}_model_1.csv"
            file_path = os.path.join(folder_path, file_name)

            if not os.path.exists(file_path):
                print(f"Warning: {file_path} not found. Skipping...")
                continue

            # Read CSV
            df = pd.read_csv(file_path)

            # Filter rows where "Evaluation Type" is "Cumulative"
            df_cumulative = df[df["Evaluation Type"] == "Cumulative"].copy()

            # Parse classification report and confusion matrix
            df_cumulative["Report"] = df_cumulative["Classification Report"].apply(ast.literal_eval)
            df_cumulative["Confusion"] = df_cumulative["Confusion Matrix"].apply(ast.literal_eval)

            # Extract metrics
            metrics_list = []
            for _, row in df_cumulative.iterrows():
                report = row["Report"]
                cm = row["Confusion"]
                fpr, fnr = compute_fpr_fnr(cm)

                metrics_list.append({
                    "Day": row["Day"],
                    # Micro-style F1 from P/R
                    "F1-score": 2 * (row["Precision"] * row["Recall"]) / (row["Precision"] + row["Recall"]),
                    # Accuracy from report
                    "Accuracy": report["accuracy"],
                    # Macro avg
                    "Macro Precision": report["macro avg"]["precision"],
                    "Macro Recall": report["macro avg"]["recall"],
                    "Macro F1": report["macro avg"]["f1-score"],
                    # Weighted avg
                    "Weighted Precision": report["weighted avg"]["precision"],
                    "Weighted Recall": report["weighted avg"]["recall"],
                    "Weighted F1": report["weighted avg"]["f1-score"],
                    # Confusion-based metrics
                    "FPR (%)": fpr,
                    "FNR (%)": fnr
                })
            all_metrics.append(pd.DataFrame(metrics_list))

        # If we gathered any results, compute averages across iterations
        if all_metrics:
            avg_df = pd.concat(all_metrics).groupby("Day").mean().reset_index()
            final_results[(window_size, ratio)] = avg_df

            # Extract Day 5 row for ranking
            day5_row = avg_df[avg_df["Day"] == 5]
            if not day5_row.empty:
                ranking_results.append({
                    "Window Size": window_size,
                    "Ratio": ratio,
                    "Day 5 Micro F1": day5_row["F1-score"].values[0],
                    "Day 5 Accuracy": day5_row["Accuracy"].values[0],
                    "Day 5 Macro F1": day5_row["Macro F1"].values[0],
                    "Day 5 Weighted F1": day5_row["Weighted F1"].values[0],
                    "Day 5 FPR (%)": day5_row["FPR (%)"].values[0],
                    "Day 5 FNR (%)": day5_row["FNR (%)"].values[0]
                })

# Display results (all days, averaged over iterations)
for key, df in final_results.items():
    window_size, ratio = key
    print(f"\n=== Transformer - Window Size: {window_size}, Ratio: {ratio} ===")
    print(df.to_string(index=False))

# Build leaderboard DataFrame
df_ranking = pd.DataFrame(ranking_results)
df_ranking_sorted = df_ranking.sort_values(by="Day 5 Macro F1", ascending=False).reset_index(drop=True)

# Print leaderboard
print("\n=== Transformers Leaderboard ranked by Day 5 Cumulative Macro F1-score ===\n")
print(df_ranking_sorted.to_string(index=False))

In [ ]:
import os
import pandas as pd
import ast  # to parse the Classification Report and Confusion Matrix safely

# Folder containing the TCN experiment files
folder_path = "Window_size_and_Ratio_experiments_TCN"

# Define parameter ranges
window_sizes = range(1, 11)  # 1 to 10
ratios = [round(i * 0.1, 1) for i in range(6)]  # 0.0 to 0.5
iterations = range(1, 4)  # 1 to 3

# Dictionary to store per-day averages
final_results = {}
# List to store Day 5 cumulative averages for ranking
ranking_results = []

def compute_fpr_fnr(cm):
    """Compute False Positive Rate and False Negative Rate in % from confusion matrix"""
    tn, fp = cm[0]
    fn, tp = cm[1]
    fpr = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0
    fnr = (fn / (fn + tp)) * 100 if (fn + tp) > 0 else 0
    return fpr, fnr

# Loop over window sizes
for window_size in window_sizes:
    for ratio in ratios:
        all_metrics = []  # Store metrics for averaging

        # Loop over iterations
        for iteration in iterations:
            file_name = f"experiments_{window_size}_{ratio}_{iteration}_model_3.csv"
            file_path = os.path.join(folder_path, file_name)

            if not os.path.exists(file_path):
                print(f"Warning: {file_path} not found. Skipping...")
                continue

            # Read CSV
            df = pd.read_csv(file_path)

            # Filter rows where "Evaluation Type" is "Cumulative"
            df_cumulative = df[df["Evaluation Type"] == "Cumulative"].copy()

            # Parse classification report and confusion matrix
            df_cumulative["Report"] = df_cumulative["Classification Report"].apply(ast.literal_eval)
            df_cumulative["Confusion"] = df_cumulative["Confusion Matrix"].apply(ast.literal_eval)

            # Extract metrics
            metrics_list = []
            for _, row in df_cumulative.iterrows():
                report = row["Report"]
                cm = row["Confusion"]
                fpr, fnr = compute_fpr_fnr(cm)

                metrics_list.append({
                    "Day": row["Day"],
                    # Micro-style F1 (from Precision/Recall)
                    "F1-score": 2 * (row["Precision"] * row["Recall"]) / (row["Precision"] + row["Recall"]),
                    # Accuracy
                    "Accuracy": report["accuracy"],
                    # Macro avg
                    "Macro Precision": report["macro avg"]["precision"],
                    "Macro Recall": report["macro avg"]["recall"],
                    "Macro F1": report["macro avg"]["f1-score"],
                    # Weighted avg
                    "Weighted Precision": report["weighted avg"]["precision"],
                    "Weighted Recall": report["weighted avg"]["recall"],
                    "Weighted F1": report["weighted avg"]["f1-score"],
                    # Confusion-based
                    "FPR (%)": fpr,
                    "FNR (%)": fnr
                })
            all_metrics.append(pd.DataFrame(metrics_list))

        # If we gathered any results, compute averages across iterations
        if all_metrics:
            avg_df = pd.concat(all_metrics).groupby("Day").mean().reset_index()
            final_results[(window_size, ratio)] = avg_df

            # Extract Day 5 row for ranking
            day5_row = avg_df[avg_df["Day"] == 5]
            if not day5_row.empty:
                ranking_results.append({
                    "Window Size": window_size,
                    "Ratio": ratio,
                    "Day 5 Micro F1": day5_row["F1-score"].values[0],
                    "Day 5 Accuracy": day5_row["Accuracy"].values[0],
                    "Day 5 Macro F1": day5_row["Macro F1"].values[0],
                    "Day 5 Weighted F1": day5_row["Weighted F1"].values[0],
                    "Day 5 FPR (%)": day5_row["FPR (%)"].values[0],
                    "Day 5 FNR (%)": day5_row["FNR (%)"].values[0]
                })

# Display results (all days averaged over iterations)
for key, df in final_results.items():
    window_size, ratio = key
    print(f"\n=== TCN - Window Size: {window_size}, Ratio: {ratio} ===")
    print(df.to_string(index=False))

# Build leaderboard DataFrame
df_ranking = pd.DataFrame(ranking_results)
df_ranking_sorted = df_ranking.sort_values(by="Day 5 Macro F1", ascending=False).reset_index(drop=True)

# Print leaderboard
print("\n=== TCN Leaderboard ranked by Day 5 Cumulative Macro F1-score ===\n")
print(df_ranking_sorted.to_string(index=False))
